In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    model="local-llm",
    base_url="http://localhost:8080/v1",
    temperature=0.7
)
agent = create_agent(
    model=chat_model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=chat_model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user initiated a series of exploratory questions about a fictional location named "Lunapolis," testing the AI\'s ability to provide specific, fabricated details regarding its geography, climate, population, and social dynamics.\n\n## SUMMARY\nThe conversation established a fictional setting called Lunapolis. Key details gathered include:\n*   **Location:** Lunapolis (the capital of the moon).\n*   **Climate:** Extreme weather, with high temperatures of 120C and low temperatures of -100C.\n*   **Population/Economy:** There are 100,000 cheese miners living in the city.\n*   **Social Conflict:** The cheese miners are unhappy with the new president, leading to a prediction that their union will strike.\n\nThe AI successfully maintained the fictional narrative and provided consistent, albeit fabricated, responses to the user\'s increasingly specific inquiries.\n\n## ARTIFACTS\nNone\n

In [ ]:
print(response["messages"][0].content)

## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

In [ ]:
print(response["messages"][-1].content)